In [ ]:
!pip install pandas numpy scikit-learn tensorflow matplotlib xgboost

import pandas as pd
import numpy as np
import zipfile
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

In [ ]:
with zipfile.ZipFile('Solar_Energy_Generation.csv.zip', 'r') as zip_ref:
    zip_ref.extractall()

with zipfile.ZipFile('Weather_Data_reordered_all.csv.zip', 'r') as zip_ref:
    zip_ref.extractall()

solar = pd.read_csv('Solar_Energy_Generation.csv')
weather = pd.read_csv('Weather_Data_reordered_all.csv')
site = pd.read_csv('Solar_Site_Details.csv')

In [ ]:
solar['Timestamp'] = pd.to_datetime(solar['Timestamp'])
weather['Timestamp'] = pd.to_datetime(weather['Timestamp'])

df = solar.merge(weather, on=['Timestamp', 'CampusKey'])
df = df.merge(site, on='SiteKey')

# 🔥 Normalize target
df['target'] = df['SolarGeneration'] / df['kWp']

df = df.sort_values('Timestamp')

In [ ]:
df['hour'] = df['Timestamp'].dt.hour
df['dayofyear'] = df['Timestamp'].dt.dayofyear

df['sin_hour'] = np.sin(2*np.pi*df['hour']/24)
df['cos_hour'] = np.cos(2*np.pi*df['hour']/24)

df['sin_day'] = np.sin(2*np.pi*df['dayofyear']/365)
df['cos_day'] = np.cos(2*np.pi*df['dayofyear']/365)

df['is_day'] = ((df['hour'] >= 6) & (df['hour'] <= 18)).astype(int)

In [ ]:
df['lag_1'] = df['target'].shift(1)
df['lag_2'] = df['target'].shift(2)
df['lag_3'] = df['target'].shift(3)
df['lag_6'] = df['target'].shift(6)

df['rolling_mean_3'] = df['target'].rolling(3).mean()
df['rolling_std_3'] = df['target'].rolling(3).std()

In [ ]:
df = df.dropna()
df = df[df['target'] < df['target'].quantile(0.99)]

In [ ]:
features = [
    'AirTemperature', 'RelativeHumidity', 'WindSpeed',
    'sin_hour', 'cos_hour',
    'sin_day', 'cos_day',
    'is_day',
    'lag_1', 'lag_2', 'lag_3', 'lag_6',
    'rolling_mean_3', 'rolling_std_3'
]

X = df[features]
y = df['target']

In [ ]:
scaler_X = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X)

scaler_y = MinMaxScaler()
y_scaled = scaler_y.fit_transform(y.values.reshape(-1,1))

In [ ]:
split = int(0.8 * len(X))

X_train_xgb, X_test_xgb = X[:split], X[split:]
y_train_xgb, y_test_xgb = y[:split], y[split:]

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8
)

xgb_model.fit(X_train_xgb, y_train_xgb)

xgb_pred = xgb_model.predict(X_test_xgb)

In [ ]:
def create_sequences(X, y, seq_length=48):
    Xs, ys = [], []
    for i in range(len(X) - seq_length):
        Xs.append(X[i:i+seq_length])
        ys.append(y[i+seq_length])
    return np.array(Xs), np.array(ys)

X_seq, y_seq = create_sequences(X_scaled, y_scaled, 48)

In [ ]:
split_seq = int(0.8 * len(X_seq))

X_train, X_test = X_seq[:split_seq], X_seq[split_seq:]
y_train, y_test = y_seq[:split_seq], y_seq[split_seq:]

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.LSTM(128, return_sequences=True),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0005),
    loss=tf.keras.losses.Huber(),
    metrics=['mae']
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(patience=5, restore_best_weights=True)

model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop]
)

In [ ]:
lstm_pred_scaled = model.predict(X_test)

lstm_pred = scaler_y.inverse_transform(lstm_pred_scaled)
y_test_actual = scaler_y.inverse_transform(y_test)

In [ ]:
min_len = min(len(xgb_pred), len(lstm_pred))

xgb_pred = xgb_pred[-min_len:]
lstm_pred = lstm_pred[-min_len:].flatten()
y_actual = y_test_actual[-min_len:].flatten()

In [ ]:
final_pred = 0.6 * xgb_pred + 0.4 * lstm_pred

In [ ]:
rmse = np.sqrt(mean_squared_error(y_actual, final_pred))
mae = mean_absolute_error(y_actual, final_pred)

print("HYBRID RMSE:", rmse)
print("HYBRID MAE:", mae)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(y_actual[:200], label='Actual')
plt.plot(final_pred[:200], label='Hybrid Predicted')
plt.legend()
plt.title("Hybrid Model Prediction")
plt.show()